# Debugging Methods Using Random Instance Generation

In this document, we introduce a debugging method for mathematical models written in JijModeling using {py:meth}`Problem.generate_random_instance <jijmodeling.Problem.generate_random_instance>`. Note that this method is specifically intended to find bugs in each element of a mathematical model (placeholders, decision variables, and the expressions composed of them).

## What is `generate_random_instance`?

{py:meth}`Problem.generate_random_instance <jijmodeling.Problem.generate_random_instance>` is a method that randomly generates instance data according to the placeholder definitions in a mathematical model, compiles the model together with the generated data, and outputs an OMMX instance. For detailed usage, see [here](./generation).

## Debugging Method Using `generate_random_instance`

Let us see how to debug a mathematical model using the graph coloring problem as an example. For the formulation of the graph coloring problem, see [here](https://jij-inc-jijzept-tutorials-en.readthedocs-hosted.com/en/latest/src/08_graph_coloring.html). First, let us intentionally create a "bad formulation".

In [1]:
import jijmodeling as jm

In [2]:
@jm.Problem.define("Graph Coloring Problem (Bad Modeling)")
def gcp_bad_modeling(problem: jm.DecoratedProblem):
    V = problem.Natural(description="Number of vertices")
    E = problem.Graph(description="Edges of the graph")
    N = problem.Natural(description="Number of colors")
    x = problem.BinaryVar(shape=(V, N), description="Binary variables for vertex-color assignment")

    problem += problem.Constraint(
        "one-color",
        [jm.sum(x[v, n] for n in N) == 1 for v in V]
    )

    problem += jm.sum(x[e[0], n] * x[e[1], n] for n in N for e in E )

gcp_bad_modeling

Problem(name="Graph Coloring Problem (Bad Modeling)", sense=MINIMIZE, objective=sum(set(N.flat_map(lambda (n: natural): E.map(lambda (e: Tuple[natural, natural]): (n, e)))).map(lambda ((n, e): Tuple[natural, Tuple[natural, natural]]): x[e[0], n] * x[e[1], n])), constraints={one-color: [Constraint(name="one-color", , lambda v: sum(set(N).map(lambda (n: natural): x[v, n])) == 1, domain=set(V)),],})

What is wrong with this formulation? At first glance, the objective function and constraints in the $\LaTeX$ display look exactly as shown in the [graph coloring formulation](https://jij-inc-jijzept-tutorials-en.readthedocs-hosted.com/en/latest/src/08_graph_coloring.html#id4), so it may seem that there is no bug.
Now, let us run {py:meth}`Problem.generate_random_instance <jijmodeling.Problem.generate_random_instance>` multiple times as follows.

In [3]:
try:
    for _ in range(50):
        _ = gcp_bad_modeling.generate_random_instance()
except jm.ModelingError as e:
    print(e)

Traceback (most recent last):
    while evaluating problem `Problem(name="Graph Coloring Problem (Bad Modeling)", sense=MINIMIZE, objective=sum(set(N.flat_map(lambda (n: natural): E.map(lambda (e: Tuple[natural, natural]): (n, e)))).map(lambda ((n, e): Tuple[natural, Tuple[natural, natural]]): x[e[0], n] * x[e[1], n])), constraints={one-color: [Constraint(name="one-color", , lambda v: sum(set(N).map(lambda (n: natural): x[v, n])) == 1, domain=set(V)),],})',
        defined at File "/var/folders/mg/mg6st30d18s7pxjjrk6pkxym0000gn/T/ipykernel_48898/3783062234.py", line 1, col 2-60
    while evaluating objective `sum(set(N.flat_map(lambda (n: natural): E.map(lambda (e: Tuple[natural, natural]): (n, e)))).map(lambda ((n, e): Tuple[natural, Tuple[natural, natural]]): x[e[0], n] * x[e[1], n]))',
        defined at File "/var/folders/mg/mg6st30d18s7pxjjrk6pkxym0000gn/T/ipykernel_48898/3783062234.py", line 13, col 16-70
    while evaluating expression `sum(set(N.flat_map(lambda (n: natural): E.

As shown above, a `ModelingError` occurs, revealing that there are cases where values outside the index range $V \times N$ of decision variable $x$ are specified. Why does this happen? Because the possible range of $e[0]$ or $e[1]$ becomes the entire set of natural numbers $\mathbb{N}$.
To fix this bug (the problematic part) in the mathematical model, you need to restrict the possible values of $e[0]$ and $e[1]$ to $V$ instead of all natural numbers. Specifically, you can remove this bug by changing the code as follows.

In [4]:
@jm.Problem.define("Graph Coloring Problem (Good Modeling)")
def gcp_good_modeling(problem: jm.DecoratedProblem):
    V = problem.Natural(description="Number of vertices")
    # Fix: specify dtype=V to explicitly restrict the possible value range
    E = problem.Graph(dtype=V, description="Edges of the graph")
    N = problem.Natural(description="Number of colors")
    x = problem.BinaryVar(shape=(V, N), description="Binary variables for vertex-color assignment")

    problem += problem.Constraint(
        "one-color",
        [jm.sum(x[v, n] for n in N) == 1 for v in V]
    )

    problem += jm.sum(x[e[0], n] * x[e[1], n] for n in N for e in E )

gcp_good_modeling

Problem(name="Graph Coloring Problem (Good Modeling)", sense=MINIMIZE, objective=sum(set(N.flat_map(lambda (n: natural): E.map(lambda (e: Tuple[V, V]): (n, e)))).map(lambda ((n, e): Tuple[natural, Tuple[V, V]]): x[e[0], n] * x[e[1], n])), constraints={one-color: [Constraint(name="one-color", , lambda v: sum(set(N).map(lambda (n: natural): x[v, n])) == 1, domain=set(V)),],})

Now, let us verify again whether this fix has removed the bug.

In [5]:
for _ in range(50):
    _ = gcp_good_modeling.generate_random_instance()

This runs {py:meth}`Problem.generate_random_instance <jijmodeling.Problem.generate_random_instance>` without errors. This result indicates that errors are less likely to occur even when randomly injecting instance data that follows the placeholder definitions, in other words, that there are fewer bugs in the model definition.

In this way, by using {py:meth}`Problem.generate_random_instance <jijmodeling.Problem.generate_random_instance>`, you can remove bugs in model definitions and improve the quality of your mathematical models.

## Notes

JijModeling's expressive power is still evolving. Therefore, it is not always possible to make {py:meth}`Problem.generate_random_instance <jijmodeling.Problem.generate_random_instance>` pass only by changing the mathematical model. In such cases, we recommend adjusting the range of placeholder values generated by {py:meth}`Problem.generate_random_instance <jijmodeling.Problem.generate_random_instance>` according to the documentation [here](./generation).

:::{admonition} For Those Who Need Help
:class: tip

If you are struggling with formulation or debugging of mathematical models in JijModeling, please make use of our [Discord community](https://discord.gg/34WkHwvY3Y).
:::